# Skincare and Cosmetics Product Recommender (MP-A10)
**Course:** CS467 - Advanced Artificial Intelligence | Duy Tan University | 2026-2027  
**Student Name:** Quách Tú Mỹ  
**Student ID:** 29201153781  
**Section:** CS467A (No. 10)  
**Instructor:** Puri Vikram  
**Domain:** Beauty and Personal Care E-Commerce

---
### Project Overview
This notebook demonstrates an end-to-end recommender system tailored for skincare products. The system combines:
1. **Domain-Specific NLP Preprocessing & TF-IDF Feature Representation**
2. **Customer Review Sentiment Analysis (Lexicon-based)**
3. **Three Recommendation Approaches:**
   - Popularity Baseline (Bayesian weighted rating)
   - NLP Content-Based Recommender (Cosine similarity over TF-IDF)
   - Sentiment-Boosted Content-Based Hybrid Recommender
4. **Three Real-World Test Cases (Query search, Item similarity, Cold-start)**
5. **Offline Train/Test Evaluation (Precision@5, Recall@5, HitRate@5, RMSE, MAE)**

## 1. Environment Setup & Data Loading

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure src is in Python path
sys.path.append(os.path.abspath(".."))
from src.nlp_pipeline import SkincareNLPPipeline, compute_sentiment_score
from src.recommender import (
    PopularityBaselineRecommender,
    NLPContentBasedRecommender,
    SentimentHybridRecommender
)
from src.evaluate import create_train_test_split, evaluate_recommenders

df_items = pd.read_csv("../data/items.csv")
df_ratings = pd.read_csv("../data/ratings.csv")

n_users = df_ratings["user_id"].nunique()
print(f"Loaded {len(df_items)} products and {len(df_ratings)} user ratings across {n_users} unique shoppers.")
display(df_items.head(3))
display(df_ratings.head(3))

## 2. NLP Pipeline: Preprocessing, TF-IDF Vectorization & Sentiment Analysis

In [ ]:
nlp = SkincareNLPPipeline()
tfidf_matrix = nlp.fit_transform(df_items)
print(f"Fitted TF-IDF vocabulary size: {len(nlp.feature_names)} features")
print(f"Feature matrix shape: {tfidf_matrix.shape}")

# Review Sentiment Demonstration
sample_reviews = [
    "The most soothing gentle hydrating moisturizer for dry sensitive skin. Pure relief from irritation.",
    "Sadly this niacinamide serum caused small irritation bumps on my cheeks after a few days."
]
for r in sample_reviews:
    score = compute_sentiment_score(r)
    print(f"Review: '{r}'")
    print(f"Calculated Sentiment Polarity: {score}\n")

## 3. Fit Recommendation Engines
Initialize and fit the Popularity Baseline, NLP Content-Based Recommender, and Sentiment-Boosted Hybrid Recommender.

In [ ]:
# 1. Popularity Baseline
pop_rec = PopularityBaselineRecommender(min_ratings=2)
pop_rec.fit(df_ratings, df_items)

# 2. NLP Content-Based
content_rec = NLPContentBasedRecommender(nlp)
content_rec.fit(df_items, df_ratings)

# 3. Sentiment Hybrid Recommender
hybrid_rec = SentimentHybridRecommender(content_rec, nlp)
hybrid_rec.fit(df_ratings, df_items)
print("All 3 recommenders successfully trained!")

## 4. Test Case 1: Need Query Search
Query: `gentle hydrating moisturizer for dry sensitive skin`

In [ ]:
query_1 = "gentle hydrating moisturizer for dry sensitive skin"
recs_1 = hybrid_rec.recommend_by_query(query_1, top_k=5)
pd.DataFrame(recs_1)[["item_id", "product_name", "category", "score", "content_similarity", "sentiment_score", "avg_rating", "top_keywords"]]

## 5. Test Case 2: Similar Product Recommendation (Item-to-Item)
Target item: `P026: Aqua UV Daily Sun Water Gel SPF50+ PA++++`

In [ ]:
target_sunscreen = "P026"
recs_2 = content_rec.recommend_similar_items(target_sunscreen, top_k=5)
pd.DataFrame(recs_2)[["item_id", "product_name", "category", "score", "skin_type_tags", "top_keywords"]]

## 6. Test Case 3: Cold-Start Item Addition & Recommendation
Add newly released serum with description only and verify that NLP can retrieve it for a relevant query.

In [ ]:
new_product = {
    "item_id": "P041",
    "product_name": "Phyto-Glow Brightening Barrier Serum",
    "category": "Serum",
    "skin_type_tags": "Dull, Uneven Skin Tone, Sensitive, Dehydrated",
    "description": "Advanced antioxidant radiance serum with 10% Ascorbic Acid, Ferulic Acid, and Hyaluronic Acid for brightening hyperpigmentation and smoothing rough skin texture.",
    "key_ingredients": "Ascorbic Acid, Ferulic Acid, Hyaluronic Acid, Vitamin E, Alpha-Arbutin"
}
content_rec.add_cold_start_item(new_product)
print(f"Dynamically added {new_product['item_id']}: {new_product['product_name']}")

cold_query = "antioxidant brightening serum with ascorbic acid and ferulic acid"
recs_3 = content_rec.recommend_by_query(cold_query, top_k=5)
pd.DataFrame(recs_3)[["item_id", "product_name", "category", "score", "top_keywords"]]

## 7. Offline Evaluation & Performance Comparison
Evaluate across 20% holdout split per user on Precision@5, Recall@5, HitRate@5, RMSE, and MAE.

In [ ]:
df_train, df_test = create_train_test_split(df_ratings, test_ratio=0.20, random_state=42)
eval_results = evaluate_recommenders(df_train, df_test, df_items, top_k=5)
df_eval = pd.DataFrame(eval_results).T
display(df_eval)

## 8. Visualizations
Display dataset distributions and recommender performance comparison charts.

In [ ]:
from IPython.display import Image
display(Image(filename="../results/rating_category_distribution.png"))
display(Image(filename="../results/recommender_evaluation_comparison.png"))